In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from pyspark.sql.functions import col, lit, to_json, get_json_object, max
from schema.bronze import stations_bronze_schema
import logging
from utils.shared import load_table, update_table, write_to_table, get_secret, set_task_value
from logger.custom_logging import set_up_logger, get_job_logger
from utils.bronze import add_buffer, generate_grid, get_all_stations_data, convert_to_json_string, add_bronze_stations_metadata
from utils.observability import create_control_table, get_last_processed_timestamp, insert_control_record


try:
    run_id = spark.conf.get("spark.databricks.clusterUsageTags.runId", 
                           spark.conf.get("pipeline_run_id", "unknown"))
except Exception:
    run_id = "unknown"

log_info = {
    "layer":"bronze",
    "job": "extract_station_data",
    "dataset": "openchargemap_api"
}

logger = set_up_logger()

log = get_job_logger(logger, **log_info, run_id=run_id)

log(logging.INFO, "Ingestion started")

# The variables below are for the control table which is used to track the last processed timestamp and observe run metrics 

start_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
pipeline = "project_voltstream"
layer = f"{log_info["layer"]}.{log_info["job"]}"
error_message = None
records_processed = 0
records_failed = 0
last_processed_timestamp = None

try:
    control_table = create_control_table(spark)
    last_timestamp = add_buffer(get_last_processed_timestamp(spark ,control_table, pipeline, layer))
    grid = generate_grid(40.4874, -74.2591, 40.9176, -73.6995)
    api_key = get_secret(scope="voltstream", key="ocm_api") 
    all_stations = get_all_stations_data(grid, last_timestamp, api_key, **log_info)
    if all_stations is None:
        set_task_value(key='my_value', value='skipped')
        status = "skipped"
    else:    
        json_string = convert_to_json_string(all_stations)
        df = spark.createDataFrame([(s, ) for s in json_string], ["raw_text"])
        df = add_bronze_stations_metadata(df)
        write_to_table(df, "bronze_dev.electrovolt.ev_stations", "append", **log_info)
        log(logging.INFO, "Ingestion completed")
        status = "success"
        last_processed_timestamp = df.agg(max("ingest_timestamp")).collect()[0][0]
        records_processed = df.count()
        set_task_value(key="my_value", value=status)
    
except Exception as e:
    status = "failed"
    error_message = str(e) 
    raise 

end_time = spark.sql("SELECT current_timestamp()").collect()[0][0] 
insert_control_record(spark, control_table, pipeline, layer, last_processed_timestamp, records_processed, records_failed, start_time, end_time, error_message,  status)